# 09 FS3 Feature Family Plan

`FS3` is reserved for causal exogenous feature families added gradually on top of the `FS2` foundation.

The active policy is:
- only models shortlisted after `FS2` continue
- retuning is mandatory because the feature space changes materially
- feature families should be introduced gradually rather than as one large buffet
- examples include load forecast, wind forecast, solar forecast, net load, and selected neighboring market information only when causally available


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig
from hourly_da.core.external_features import build_external_family_catalog, load_external_feature_store
from hourly_da.core.methodology import (
    STARTER_ENDOGENOUS_FEATURE_NOTE,
    feature_stage_policy_frame,
    model_status_frame,
    shortlisting_policy_frame,
)
from hourly_da.core.reporting import find_latest_run, load_csv, load_json
from hourly_da.core.tuning import build_tuning_placeholder, tuning_cadence_frame, tuning_snippet_frame
from hourly_da.notebook_support import estimate_run_duration_seconds, format_duration, load_selected_case_weeks

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root


def latest_run_or_none(run_label: str) -> Path | None:
    try:
        return find_latest_run(output_root, run_label)
    except FileNotFoundError:
        return None


In [ ]:
display(
    feature_stage_policy_frame()
    .loc[lambda df: df["fs_level"] == "FS3"]
    .reset_index(drop=True)
)


In [ ]:
display(shortlisting_policy_frame())
display(tuning_cadence_frame().loc[lambda df: df["fs_level"] == "FS3"].reset_index(drop=True))
display(tuning_snippet_frame(fs_level="FS3"))


In [ ]:
try:
    external_store = load_external_feature_store(config)
    external_catalog = build_external_family_catalog(config, external_store)
    display(
        external_catalog[
            [
                "family_name",
                "availability_class",
                "lead_scope",
                "allowed_usage_mode",
                "current_usage_mode",
                "current_issue_status",
            ]
        ].reset_index(drop=True)
    )
except FileNotFoundError as exc:
    print(f"External feature store is not fully available yet: {exc}")


Implementation reminder:
- this notebook is planning-oriented
- the next execution step later is to promote `FS2` survivors into gradual causal-family tests
- no `FS3` backtests are launched here
